# 3D Visual Grounding — full experiment runner

Runs **every** experiment for the IMAVIS revision from a single notebook. It is built for
**Run All**: nothing below raises on a machine that cannot do the work, it just says so and
moves on.

## The three phases

Experiments are grouped by what they *cost* and what they *need*, because those are the
two things that decide whether you can run one right now.

| phase | what it does | needs | cost |
|---|---|---|---|
| **A — retraining** | trains a model from scratch, one arm per ablation | GPU | hours **each** |
| **B — evaluation** | runs a trained checkpoint over val | GPU + a checkpoint | minutes |
| **C — post-processing** | reads files off disk; builds no model | nothing | seconds to minutes |

Phase A comes first because it is the long pole and everything downstream waits on it.
Its seven experiments are the ones the revision actually depends on:

| | experiment | ablates | reviewer |
|---|---|---|---|
| **A1** | `train-no-copypaste` | proposal copy-paste augmentation | R2, R4.7 |
| **A2** | `train-parser-gpt` | — (reference arm, variant A) | R4.4 |
| **A3** | `train-parser-spacy` | LLM → rule-based parser (variant B) | R3.1 |
| **A4** | `train-parser-llama` | GPT API → open LLaMA-3 (variant C) | R2, R3.1 |
| **A5** | `train-parser-none` | the parser entirely (variant D) | R4.4 |
| **A6** | `train-parser-smalllm` | GPT → 0.5B local model (variant E) | R3.1, R4.5 |
| **A7** | `train-seeds` | nothing — repeats A2 under 2 more seeds | R4.8 |

Each has **its own function, its own cell and its own switch**, so a Colab session that
dies in the middle of A4 can be resumed by turning A1–A3 off and running again.

## What decides whether something executes

| | |
|---|---|
| **its switch** | one `RUN_*` flag per experiment in the config cell |
| **where you are** | Colab vs a local machine, detected automatically |
| **what you have** | whether a CUDA kernel really launches — not `is_available()`, which lies |
| **what is already done** | every experiment declares the files it produces; existing ones are skipped |

Override any of it in the config cell: `FORCE_GPU` runs GPU work anyway (at
`MIN_BATCH_SIZE`, so it does not take the machine down), `OVERRIDE_EXP` re-runs experiments
whose outputs already exist.

All code lives under `experiments/`:

```
experiments/ablation/      retrain or rebuild a cache      (parsers/, runners/)
experiments/analysis/      post-hoc, reads predictions.p   — CPU, no model
experiments/complexity/    FLOPs, memory, latency
experiments/diagnostics/   does the implementation match the paper?
```

Variant lettering used throughout: **A** = GPT-4o-mini (paper main), **B** = spaCy,
**C** = LLaMA-3, **D** = no parser, **E** = small local LM.

## 1 — configuration

The only cell you need to edit.

In [ ]:
# ======================================================================================
# USER CONFIGURATION
# ======================================================================================

DEVICE       = "cpu"     # "auto" | "cpu" | "cuda"   -- auto probes for a working GPU
GPU_ID       = "0"

FORCE_GPU    = False      # run GPU experiments even without a usable GPU (very slow)
OVERRIDE_EXP = True       # re-run experiments whose output files already exist.
                          # Keep False: every expensive artefact (the small-LM parse
                          # cache above all -- 80 min on a T4) is written to disk once
                          # and then reused offline. True regenerates all of it.
MIN_BATCH_SIZE = 1        # batch size substituted when FORCE_GPU runs GPU work on CPU

# Host RAM, not GPU memory. ScannetReferenceDataset precomputes a GloVe embedding matrix
# for every annotation up front (~22 GB train + ~7 GB val, float64), which SIGKILLs the
# process before epoch 1 on any Colab tier. CachedSceneDataset builds them per annotation
# instead: the full train split then costs ~1.8 GB and builds in ~3s. Set False only to
# reproduce the original eager behaviour on a machine with ~40 GB of RAM to spare.
#
# Measured, so there is no third "partial preload" mode: building one annotation's
# embeddings costs 0.06 ms, which is 0.3% of the ~20 ms __getitem__ spends -- the rest is
# point-cloud work (subsampling, unique, reductions). Precomputing them to disk would
# cost 11 GB (float32) to save 0.3%, and reading 591 KB per item back would be slower
# than recomputing it. The loading win is in NUM_WORKERS below, not in preloading.
LAZY_LANG_DATA = True

# ---- data loading ----------------------------------------------------------------------
# Background worker processes that prepare batches ahead of the main one, so loading
# overlaps GPU compute instead of blocking it. This is the actual speedup: measured on an
# 8-core machine, 0 -> 4 workers took loading from 13.6 to 6.2 min per epoch (2.2x).
#
# None = auto (cpu_count - 1, capped at 4). 0 restores serial loading. Each worker holds
# its own copy of the dataset, so RAM grows with worker count -- affordable only because
# LAZY_LANG_DATA keeps that copy near 1.8 GB. If Colab reports OOM, lower this first.
NUM_WORKERS     = 0      # None = auto; 0 = serial; or an explicit count
PREFETCH_FACTOR = 1      # batches each worker keeps ready in RAM

# Frozen-detector protocol. When True (and the cache is present and complete) the
# training and evaluation commands get --use_cached_scenes, which makes
# experiments/ablation/ablation_config.apply() set no_detection=True and swap RefNet for
# CachedRefNet -- the detector is then not constructed at all.
USE_CACHED_SCENES  = True
CACHED_SCENES_ROOT = "cached_scenes"

# Trained run under outputs/ that evaluation and analysis read.
CHECKPOINT = "2024-12-18_20-40-38_3DVG-FIXED"

# Which MatchModule to build when loading CHECKPOINT. That run was trained with the
# older fusion head ("models/match_module original.py"); models/match_module.py has since
# added a post-graph attention stage, four more GCN layers and a third attention layer,
# so loading it into the current module would leave 178 tensors randomly initialised.
# ablation_hooks now refuses that outright rather than reporting a meaningless accuracy.
# Set "current" only for a checkpoint trained with today's architecture.
CHECKPOINT_FUSION_VARIANT = "original"

EPOCH      = 20          # see the training section below for why 50 and not 100
BATCH_SIZE = 16
SEEDS      = [1, 2]
EXTRA_ARGS = ["--use_color", "--use_normal"]     # MUST match how the scene cache was built

# ======================================================================================
# TRAINING -- warm start and hyper-parameters.
#
# These apply to every phase-A arm. They are written into the runners by the
# "apply training configuration" cell further down, so editing them here is enough --
# you do not need to open seven runner files.
# ======================================================================================

# ---- warm start ----------------------------------------------------------------------
# Load fusion weights from an already-trained run instead of starting from random
# initialisation. Under the frozen-detector protocol with FUSION_VARIANT="original" the
# checkpoint covers the model completely -- 146 of 146 tensors -- so this is a true
# fine-tune, not a partial one, and it is where most of the speedup comes from.
#
# Every arm warm-starts from the SAME checkpoint. That is what keeps the comparison
# fair: the arms share a starting point and differ only in what each one ablates.
# Disclose it in the manuscript -- these are fine-tuned runs, not from-scratch runs.
#
# Set False for A7 (seeds) if you want run-to-run variance measured from random init;
# see the note in the phase-A seeds cell.
WARM_START      = True
WARM_START_FROM = CHECKPOINT     # the 2024-12-18 run; the only one present in outputs/
WARM_START_FUSION_VARIANT = "original"   # the head WARM_START_FROM was trained with

# ---- hyper-parameters ------------------------------------------------------------------
# VAL_STEP is in ITERATIONS, not epochs (lib/solver.py checks _global_iter_id % val_step).
# The value that used to be hardcoded was 10 -- a full validation pass every 10
# iterations, which costs more than the training it interrupts. 5000 puts validation at
# roughly a couple of times per epoch at batch size 8.
VAL_STEP     = 50      # validate every N iterations
VERBOSE      = 50        # print a training line every N iterations
LR           = 0.002     # initial learning rate
COSLR        = True      # cosine learning-rate schedule
LANG_NUM_MAX = 32        # language samples per scene per batch

# Parse caches. Keys label the reports; values are folders under data_parsing/.
PARSERS = {
    "gpt4o-mini": "final_parsing_tokenized",           # A - paper's main configuration
    "spacy":      "spacy_parsing_tokenized",           # B - rule-based
    "llama":      "llama_parsing_tokenized_clipped",   # C - LLaMA-3
    "none":       "noparse_tokenized",                 # D - no parser
    "smalllm":    "smalllm_parsing_tokenized",         # E - small local LM
}

CORRUPTION_RATES = [0.10, 0.25, 0.50]
CORRUPTION_MODE  = "all"
SMALLLM_MODEL    = "qwen2.5"     # see --list-models on run_smalllm_parser.py
ANNOTATION_N     = 200

BASELINE_PREDICTIONS = {
    "3DVG-Trans": "outputs/3DVG-TRANS-outputs/predictions.p",
}

# ======================================================================================
# PHASE A -- RETRAINING.  The seven experiments that train a model from scratch.
#
# These are the expensive ones and the ones the revision depends on: each is hours of
# GPU time, and each has its own switch so a run can be resumed after a Colab timeout
# without repeating what already finished. Turn off what is already trained.
#
# Every arm is identical except for the one thing it ablates, and every arm trains only
# the fusion head (frozen detector), so a difference between two of them is attributable
# to that one change.
# ======================================================================================

RUN_TRAIN_NO_COPYPASTE      = False   # A1  proposal copy-paste off        (R2, R4.7)
RUN_TRAIN_PARSER_GPT        = False   # A2  variant A - GPT-4o-mini        (reference arm)
RUN_TRAIN_PARSER_SPACY      = False   # A3  variant B - rule-based spaCy   (R3.1)
RUN_TRAIN_PARSER_LLAMA      = False   # A4  variant C - LLaMA-3            (R2, R3.1)
RUN_TRAIN_PARSER_NONE       = False   # A5  variant D - no parser          (R4.4)
RUN_TRAIN_PARSER_SMALLLM    = False   # A6  variant E - small local LM     (R3.1, R4.5)
RUN_TRAIN_SEEDS             = False   # A7  main config under 2 more seeds (R4.8)

# Not an ablation arm -- a hyper-parameter sweep that also retrains. Kept apart from the
# seven above because it answers a different question (how many attention layers?).
RUN_TRAIN_ATTENTION_SWEEP = False    # R3.3

# ======================================================================================
# PHASE B -- EVALUATION.  Needs a GPU and a trained checkpoint; trains nothing.
# ======================================================================================

RUN_EVAL_MAIN        = False   # B1  checkpoint over val -> predictions.p (feeds phase C)
RUN_EVAL_CORRUPTION  = False   # B2  corrupted parses at test time        (R4.3)
RUN_EVAL_PARSER_SWAP = False   # B3  parser swapped at test time only     (R4.4)

# ======================================================================================
# PHASE C -- POST-PROCESSING.  CPU only. Reads files off disk, builds no model.
# ======================================================================================

RUN_POST_PARSE_CACHES = False   # C1  build the parse caches (variant E needs a GPU)
RUN_POST_SCENE_CACHE  = False   # C2  rebuild the scene cache -- GPU, see the note below
RUN_POST_PARSER_ACC   = True    # C3  parser target accuracy               (R3.1)
RUN_POST_ANALYSES     = True    # C4  the paper's tables and figures
RUN_POST_COMPLEXITY   = True    # C5  FLOPs, memory, latency               (R2, R4.5)
RUN_POST_DIAGNOSTICS  = True    # C6  does the implementation match the paper?

# C2 rebuilds the scene cache from the detector checkpoint. cached_scenes.zip already
# contains a complete cache (meta.json: complete=true, 562 train + all val scenes) built
# from that same checkpoint, so it is off by default: it would spend GPU hours
# reproducing bytes that are already on disk. Turn it on only to regenerate the cache.


## 2 — environment

Detects Colab, then probes the GPU by actually launching a kernel. `torch.cuda.is_available()`
is not enough: a card older than the kernels a torch build ships reports `True` and then
fails on the first real operation.

In [ ]:
import glob as _glob
import importlib.util
import json
import os
import subprocess
import sys
import textwrap
import time
from pathlib import Path

REPO = Path.cwd()
RESULTS = {}

if not (REPO / "scripts" / "ScanRefer_train.py").is_file():
    raise SystemExit(f"Run this notebook from the repository root. Current cwd: {REPO}")

IN_COLAB = importlib.util.find_spec("google.colab") is not None

GPU_USABLE, GPU_NAME, GPU_NOTE = False, None, ""
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        try:
            _probe = torch.randn(8, 8, device="cuda")
            (_probe @ _probe).sum().item()
            GPU_USABLE = True
            GPU_NOTE = f"{GPU_NAME} (sm_{major}{minor})"
        except Exception as error:
            arch = getattr(torch.cuda, "get_arch_list", lambda: ["?"])()
            GPU_NOTE = (f"{GPU_NAME} (sm_{major}{minor}) present but NO KERNEL RUNS -- "
                        f"this torch ships {arch}. "
                        f"{type(error).__name__}: {str(error).splitlines()[0][:80]}")
    else:
        GPU_NOTE = "no CUDA device visible to torch"
except Exception as error:
    GPU_NOTE = f"torch unavailable: {error}"

if DEVICE == "auto":
    DEVICE = "cuda" if GPU_USABLE else "cpu"
elif DEVICE == "cuda" and not GPU_USABLE:
    print("[warn] DEVICE='cuda' but no kernel launches here; GPU stages still gated on "
          "FORCE_GPU.")

print("=" * 88)
print(f"environment  : {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"python       : {sys.version.split()[0]}")
print(f"GPU          : {'USABLE -- ' + GPU_NOTE if GPU_USABLE else 'UNUSABLE -- ' + GPU_NOTE}")
print(f"device       : {DEVICE}")
print(f"force_gpu    : {FORCE_GPU}     override_exp: {OVERRIDE_EXP}")
print("=" * 88)

if not GPU_USABLE and not FORCE_GPU:
    print("GPU experiments will be SKIPPED. Set FORCE_GPU=True to run them on CPU "
          f"at batch size {MIN_BATCH_SIZE} (expect many hours).")

## 3 — packages

Only acts on Colab. `pyg-lib` is the one that matters: `models/match_module.py` calls
`knn_graph` inside the forward pass, so without it **the model cannot run at all**. It is not
on PyPI — it installs only from the PyG index keyed to the exact torch build, which is why
the version is resolved at run time rather than hardcoded.

In [ ]:
def _pip(*args):
    print("$ pip install " + " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

def ensure_packages(force=False):
    if not (IN_COLAB or force):
        print("not on Colab -- skipping installs. Pass force=True to run them anyway.")
        return
    for name in ["req/requirements.txt", "req/req_env.txt"]:
        if (REPO / name).is_file():
            _pip("-r", str(REPO / name))
    try:
        import torch
        _pip("pyg_lib", "-f", f"https://data.pyg.org/whl/torch-{torch.__version__}.html")
    except Exception as error:
        print(f"[warn] could not install pyg_lib: {error}")
    try:
        import spacy  # noqa: F401
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
                       check=False)
    except Exception:
        pass

ensure_packages()

# Report what is importable, and be explicit about the one that blocks the model.
print("\n-- packages --")
MODEL_RUNNABLE = False
for module in ["torch", "numpy", "scipy", "spacy", "matplotlib", "tqdm", "transformers",
               "torch_geometric", "pyg_lib", "torch_cluster", "openai", "h5py",
               "easydict", "thop", "plyfile", "tensorboardX"]:
    try:
        loaded = importlib.import_module(module)
        print(f"  {module:18s} OK    {getattr(loaded, '__version__', '')}")
    except Exception:
        print(f"  {module:18s} --")
try:
    from torch_geometric.nn import knn_graph
    import torch as _t
    knn_graph(_t.randn(16, 3), k=4, batch=_t.zeros(16, dtype=_t.long))
    MODEL_RUNNABLE = True
    print("\n  knn_graph works -> the model can be built and run")
except Exception as error:
    print(f"\n  knn_graph FAILS -> every stage that builds the model is blocked")
    print(f"    {type(error).__name__}: {str(error)[:120]}")
    print("    fix: pip install pyg_lib -f https://data.pyg.org/whl/torch-<version>.html")

## 4 — the experiment registry

One declarative entry per experiment. `device` says what it needs, `produces` lists the files
that prove it already ran. Everything below is orchestration around these entries — no
analysis is reimplemented in the notebook, so the notebook and the command line always agree.

In [ ]:
AB = "experiments/ablation"
AN = "experiments/analysis"
CX = "experiments/complexity"
DG = "experiments/diagnostics"

def parse_cache_files(folder, splits=("train", "val")):
    return [f"data_parsing/{folder}/tokenized_parsed_result_{s}.json" for s in splits]

def have(*parts):
    return (REPO.joinpath(*parts)).exists()

CACHE_READY = have(CACHED_SCENES_ROOT, "meta.json")
if CACHE_READY:
    try:
        CACHE_READY = bool(json.loads(
            (REPO / CACHED_SCENES_ROOT / "meta.json").read_text()).get("complete"))
    except Exception:
        CACHE_READY = False

CACHE_ARGS = (["--use_cached_scenes", "--cached_scenes_root", CACHED_SCENES_ROOT]
              if (USE_CACHED_SCENES and CACHE_READY) else [])

# Every experiment belongs to exactly one phase, and every phase runs on its own terms:
# A costs GPU-hours and produces checkpoints, B needs a checkpoint but trains nothing,
# C needs neither and runs on any machine. Keeping them apart is what lets a Colab
# session that dies during phase A be resumed without re-running phase C.
PHASE_OF = {}          # stage key -> "A" | "B" | "C"
RUN = {}               # stage key -> bool, assembled from the switches in section 1

EXPERIMENTS = []

def add(key, stage, device, script, args=(), produces=(), note="", refuses=None):
    """`refuses` maps an exit code to a reason, for scripts that decline on purpose.

    A script that detects a bad precondition and stops is doing its job; reporting it
    as a crash next to genuine failures would train you to ignore the summary."""
    EXPERIMENTS.append(dict(key=key, stage=stage, device=device, script=script,
                            args=[str(a) for a in args], produces=list(produces),
                            note=note, refuses=dict(refuses or {})))

def stage(key, phase, enabled):
    """Declare a stage, its phase and its switch, in one place."""
    PHASE_OF[key] = phase
    RUN[key] = bool(enabled)

# ======================================================================================
# PHASE A -- retraining.  One stage, one function, one switch per experiment.
# ======================================================================================

TRAIN_NOTE = "GPU, hours; trains the fusion head from scratch"

stage("train_no_copypaste", "A", RUN_TRAIN_NO_COPYPASTE)
add("train-no-copypaste", "train_no_copypaste", "gpu", f"{AB}/runners/run_no_copypaste.py",
    [], [], "proposal copy-paste disabled -- isolates the augmentation (R2, R4.7)")

# The four parser arms plus the small-LM arm. Same command, same settings, one parse
# cache apiece: the only thing that differs between them is which parser produced the
# language input, which is exactly what the ablation is meant to measure.
PARSER_TRAINING = [
    ("gpt",     "A", "gpt4o-mini", "run_parser_gpt",     RUN_TRAIN_PARSER_GPT,
     "reference arm -- the paper's main configuration"),
    ("spacy",   "B", "spacy",      "run_parser_spacy",   RUN_TRAIN_PARSER_SPACY,
     "rule-based dependency parser -- is an LLM needed at all? (R3.1)"),
    ("llama",   "C", "llama",      "run_parser_llama",   RUN_TRAIN_PARSER_LLAMA,
     "open LLaMA-3 instead of the GPT API (R2, R3.1)"),
    ("none",    "D", "none",       "run_parser_none",    RUN_TRAIN_PARSER_NONE,
     "no parser, architecture untouched -- isolates the parser (R4.4)"),
    ("smalllm", "E", "smalllm",    "run_parser_smalllm", RUN_TRAIN_PARSER_SMALLLM,
     "0.5B local model -- removes the API from the critical path (R3.1, R4.5)"),
]

for _key, _letter, _parser, _script, _enabled, _why in PARSER_TRAINING:
    _stage = f"train_parser_{_key}"
    stage(_stage, "A", _enabled)
    add(f"train-parser-{_key}", _stage, "gpu", f"{AB}/runners/{_script}.py", [], [],
        f"variant {_letter}: {_why}")

stage("train_seeds", "A", RUN_TRAIN_SEEDS)
add("train-seeds", "train_seeds", "gpu", f"{AB}/runners/run_seeds.py", [], [],
    f"main configuration under seeds {SEEDS} -- run-to-run spread (R4.8)")

stage("train_attention_sweep", "A", RUN_TRAIN_ATTENTION_SWEEP)
add("train-attention-sweep", "train_attention_sweep", "gpu",
    f"{AB}/runners/sweep_attention_layers.py", [],
    ["outputs/ablation/attention_layer_sweep/sweep_summary.json"],
    "how many attention layers? hyper-parameter sweep (R3.3)")

# ======================================================================================
# PHASE B -- evaluation.  Needs a checkpoint. Builds a model, but never trains it.
# ======================================================================================

stage("eval_main", "B", RUN_EVAL_MAIN)
add("evaluate-main", "eval_main", "gpu", "scripts/ScanRefer_eval.py",
    ["--folder", CHECKPOINT, "--reference", "--force", "--lang_num_max", "1",
     "--batch_size", BATCH_SIZE, "--fusion_variant", CHECKPOINT_FUSION_VARIANT,
     *EXTRA_ARGS, *CACHE_ARGS],
    [f"outputs/{CHECKPOINT}/predictions.p"], "produces predictions.p for phase C")

stage("eval_corruption", "B", RUN_EVAL_CORRUPTION)
add("corruption-sweep", "eval_corruption", "gpu", f"{AB}/runners/run_parse_corruption.py",
    [], [f"outputs/{CHECKPOINT}/corruption"],
    "one val pass per corruption level -- how gracefully does it degrade? (R4.3)")

stage("eval_parser_swap", "B", RUN_EVAL_PARSER_SWAP)
add("eval-only-swap", "eval_parser_swap", "gpu",
    f"{AB}/runners/run_eval_only_parser_swap.py", [], [],
    "GPT-trained model fed another parser at test time (R4.4)")

# ======================================================================================
# PHASE C -- post-processing.  No training, no checkpoint. CPU unless noted.
# ======================================================================================

stage("parse_caches", "C", RUN_POST_PARSE_CACHES)
add("parse-spacy", "parse_caches", "cpu", f"{AB}/parsers/run_spacy_parser.py",
    ["--splits", "train", "val"], parse_cache_files(PARSERS["spacy"]),
    "variant B, rule-based")
add("parse-none", "parse_caches", "cpu", f"{AB}/parsers/make_noparse_cache.py",
    ["--splits", "train", "val", "--both"], parse_cache_files(PARSERS["none"]),
    "variant D, all fields 'unk'")
add("parse-corrupt", "parse_caches", "cpu", f"{AB}/parsers/corrupt_parse_cache.py",
    ["--splits", "val", "--source", PARSERS["gpt4o-mini"], "--mode", CORRUPTION_MODE,
     "--rates", *[str(r) for r in CORRUPTION_RATES]],
    [f"data_parsing/{PARSERS['gpt4o-mini']}_corrupt_{CORRUPTION_MODE}_{int(round(r*100)):02d}"
     f"/tokenized_parsed_result_val.json" for r in CORRUPTION_RATES],
    "10/25/50% corrupted val parses")
add("parse-smalllm", "parse_caches", "gpu", f"{AB}/parsers/run_smalllm_parser.py",
    ["--splits", "train", "val", "--model", SMALLLM_MODEL, "--device", DEVICE],
    parse_cache_files(PARSERS["smalllm"]), "variant E, small local LM")

stage("scene_cache", "C", RUN_POST_SCENE_CACHE)
add("scene-cache", "scene_cache", "gpu", f"{AB}/scenes_cache.py",
    ["--splits", "train", "val", *EXTRA_ARGS], [f"{CACHED_SCENES_ROOT}/meta.json"],
    "frozen detector output, ~40 min")
add("scene-cache-validate", "scene_cache", "gpu", f"{DG}/validate_scene_cache.py",
    [*EXTRA_ARGS], [], "end-to-end vs cached, must agree to 1e-4")

stage("parser_acc", "C", RUN_POST_PARSER_ACC)
for label, folder in PARSERS.items():
    add(f"target-acc-{label}", "parser_acc", "cpu",
        f"{AB}/parsers/eval_parser_target_accuracy.py",
        ["--splits", "train", "--parsed-dir", f"data_parsing/{folder}", "--tag", label],
        [f"outputs/parser_eval/target_accuracy_{label}.json"],
        f"needs data_parsing/{folder}")

PRED = f"outputs/{CHECKPOINT}/predictions.p"
SPEC = ["--predictions", f"ours={PRED}"]
for _label, _path in BASELINE_PREDICTIONS.items():
    SPEC += ["--predictions", f"{_label}={_path}"]
PARSE_ARGS = []
for _label, _folder in PARSERS.items():
    if have("data_parsing", _folder, "tokenized_parsed_result_val.json"):
        PARSE_ARGS += ["--parse", f"{_label}={_folder}"]

stage("analyses", "C", RUN_POST_ANALYSES)
add("results-table", "analyses", "cpu", f"{AN}/results_table.py", [*SPEC, "--latex"],
    ["outputs/analysis/results_table/results_table.json"], "the paper's main table")
add("linguistic-complexity", "analyses", "cpu", f"{AN}/linguistic_complexity.py", SPEC,
    ["outputs/analysis/linguistic_complexity/linguistic_complexity.md"], "R4.2")
add("parse-quality-split", "analyses", "cpu", f"{AN}/parse_quality_split.py",
    ["--predictions", PRED, *PARSE_ARGS],
    ["outputs/analysis/parse_quality_split/parse_quality_split.md"], "R4.3")
add("failure-cases", "analyses", "cpu", f"{AN}/failure_cases.py",
    ["--predictions", PRED, "--top", "6"],
    ["outputs/analysis/failure_cases/failure_cases.json"], "R2, R4")
add("failure-figures", "analyses", "cpu", f"{AN}/render_failure_figures.py",
    ["--predictions", PRED], [f"{AN}/figures/figures.json"],
    "renders the qualitative figure")
add("annotation-sheet", "analyses", "cpu", f"{AN}/annotation_sheet.py",
    ["--num", ANNOTATION_N, *PARSE_ARGS],
    ["outputs/analysis/annotation/sampling_manifest.json"], "200-sample manual sheets")
add("error-propagation", "analyses", "cpu", f"{AN}/parse_error_propagation.py",
    ["--run-dir", f"outputs/{CHECKPOINT}/corruption"],
    ["outputs/analysis/parse_error_propagation/parse_error_propagation.md"],
    "needs B2")
add("seed-aggregate", "analyses", "cpu", f"{AB}/runners/aggregate_seed_results.py",
    ["--pattern", "*ABL-SEED*", "--latex"], [], "R4.8, needs A7")

stage("complexity", "C", RUN_POST_COMPLEXITY)
add("parsing-offline-check", "complexity", "cpu", f"{CX}/measure_parsing_latency.py",
    ["--mode", "offline_check"], [], "is parsing offline? R4.5")
add("parsing-latency-spacy", "complexity", "cpu", f"{CX}/measure_parsing_latency.py",
    ["--mode", "spacy", "--num_samples", "500"], [], "per-query parse cost")
add("complexity-model", "complexity", "gpu", f"{CX}/measure_complexity.py",
    ["--variant", "both", *EXTRA_ARGS, "--latency_iters", "100", "--repeat", "2"],
    ["outputs/complexity/complexity_report.json"], "FLOPs + peak memory, R2")

stage("diagnostics", "C", RUN_POST_DIAGNOSTICS)
add("verify-eq7", "diagnostics", "cpu", f"{DG}/verify_eq7_distance_bias.py", [],
    ["outputs/diagnostics/eq7_distance_bias.json"], "R1.3 / R4.6, sign of Eq. 7")
add("audit-scene-cache", "diagnostics", "cpu", f"{DG}/audit_scene_cache.py", [],
    ["outputs/diagnostics/scene_cache_audit_val.json"],
    "model-free cache integrity + recall ceiling")
add("cached-eval-cpu", "diagnostics", "cpu", f"{DG}/cached_eval_cpu.py",
    ["--num-samples", "128", "--fusion-variant", CHECKPOINT_FUSION_VARIANT],
    ["outputs/diagnostics/cached_eval_cpu.json"],
    "cached path accuracy without CUDA",
    refuses={2: "checkpoint does not match the current MatchModule"})

# ---- report -------------------------------------------------------------------------
PHASE_NAME = {"A": "retraining", "B": "evaluation", "C": "post-processing"}
print(f"{len(EXPERIMENTS)} experiments registered across {len(RUN)} stages\n")
for _phase in ("A", "B", "C"):
    _stages = [s for s, p in PHASE_OF.items() if p == _phase]
    _exps = [e for e in EXPERIMENTS if PHASE_OF[e["stage"]] == _phase]
    _on = sum(1 for s in _stages if RUN[s])
    print(f"  phase {_phase} ({PHASE_NAME[_phase]:<15}) "
          f"{len(_exps):>2} experiment(s) in {len(_stages):>2} stage(s), {_on} switched on")
print(f"\nscene cache: "
      f"{'READY -> --use_cached_scenes will be passed' if CACHE_ARGS else 'not used'}")


## 5 — preflight

What will happen, before anything happens. Read the `action` column: `RUN` means it executes,
`SKIP-done` means its outputs are already on disk, `SKIP-gpu` means this machine cannot do it,
`SKIP-stage` means you turned the stage off, `SKIP-input` means a required input is missing.

In [ ]:
def produced(exp):
    """True when every declared output already exists (globs allowed)."""
    if not exp["produces"]:
        return False
    return all(_glob.glob(str(REPO / pattern)) for pattern in exp["produces"])

def missing_inputs(exp):
    """Cheap guards for the few experiments with a hard prerequisite."""
    key = exp["key"]
    if key.startswith("target-acc-"):
        folder = PARSERS[key[len("target-acc-"):]]
        if not have("data_parsing", folder, "tokenized_parsed_result_train.json"):
            return f"data_parsing/{folder} (train)"
    # A retraining arm cannot start without the parse cache it is defined by.
    if key.startswith("train-parser-"):
        label = key[len("train-parser-"):]
        folder = PARSERS["gpt4o-mini" if label == "gpt" else label]
        if not have("data_parsing", folder, "tokenized_parsed_result_train.json"):
            return f"data_parsing/{folder} (train) -- build it in phase C1"
    if key in {"results-table", "linguistic-complexity", "parse-quality-split",
               "failure-cases", "failure-figures", "cached-eval-cpu"} and not have(PRED):
        return PRED
    if key == "error-propagation":
        # An empty corruption/ directory exists before B2 ever runs, so existence
        # alone is not evidence -- look for the archived prediction files themselves.
        if not _glob.glob(str(REPO / "outputs" / CHECKPOINT / "corruption" / "**" / "*.p"),
                          recursive=True):
            return f"archived predictions in outputs/{CHECKPOINT}/corruption (phase B2)"
    if key == "seed-aggregate" and not _glob.glob(str(REPO / "outputs" / "*ABL-SEED*")):
        return "at least one outputs/*ABL-SEED* run (phase A7)"
    if key in {"parse-quality-split", "annotation-sheet"} and not PARSE_ARGS:
        return "at least one val parse cache"
    if key in {"audit-scene-cache", "scene-cache-validate"} and not CACHE_READY:
        return f"{CACHED_SCENES_ROOT}/meta.json with complete=true"
    if exp["device"] == "gpu" and not MODEL_RUNNABLE:
        return "pyg_lib (knn_graph) -- the model cannot be built"
    return None

def decide(exp):
    if not RUN.get(exp["stage"], False):
        return "SKIP-off", "switched off in the config cell"
    gap = missing_inputs(exp)
    if gap:
        return "SKIP-input", f"needs {gap}"
    if exp["device"] == "gpu" and not GPU_USABLE and not FORCE_GPU:
        return "SKIP-gpu", "no usable GPU (set FORCE_GPU=True to force)"
    if produced(exp) and not OVERRIDE_EXP:
        return "SKIP-done", "outputs already exist (set OVERRIDE_EXP=True to redo)"
    return "RUN", ""

PLAN = {}
PHASE_TITLE = {
    "A": "RETRAINING      -- GPU, hours each. The seven the revision depends on.",
    "B": "EVALUATION      -- GPU, needs a trained checkpoint. Trains nothing.",
    "C": "POST-PROCESSING -- reads files off disk. No training, no checkpoint.",
}

for exp in EXPERIMENTS:
    PLAN[exp["key"]] = decide(exp)[0]

for phase in ("A", "B", "C"):
    chosen = [e for e in EXPERIMENTS if PHASE_OF[e["stage"]] == phase]
    if not chosen:
        continue
    print(f"\n{'=' * 96}")
    print(f"PHASE {phase} -- {PHASE_TITLE[phase]}")
    print("=" * 96)
    print(f"  {'experiment':<24} {'stage':<24} {'dev':<4} {'action':<11} why")
    print("  " + "-" * 92)
    for exp in chosen:
        action, why = decide(exp)
        print(f"  {exp['key']:<24} {exp['stage']:<24} {exp['device']:<4} "
              f"{action:<11} {why}")
    running = sum(1 for e in chosen if PLAN[e["key"]] == "RUN")
    print(f"  -> {running} of {len(chosen)} will run")

print(f"\n{'=' * 96}")
counts = {a: sum(1 for v in PLAN.values() if v == a) for a in sorted(set(PLAN.values()))}
print("TOTAL   " + "   ".join(f"{a}: {n}" for a, n in counts.items()))
if FORCE_GPU and not GPU_USABLE:
    print(f"\n  FORCE_GPU is on without a GPU -- GPU experiments will use "
          f"batch_size={MIN_BATCH_SIZE}.")

## 6 — Google Drive mirror

Colab runtimes are disposable. Mount Drive so every report, checkpoint and figure
survives the session; `outputs/` is copied there after each experiment, not just at
the end, so a run that dies partway still leaves its finished work behind.


In [ ]:
# ======================================================================================
# Google Drive mirror
#
# A Colab runtime is disposable: when it recycles, /content is erased along with every
# checkpoint, report and figure. Anything worth keeping has to leave the machine.
#
# Mounting is the only step that needs you: Colab will open a consent prompt the first
# time. Everything after that is automatic -- outputs/ is copied to Drive after each
# experiment, so a run that dies at hour three still leaves its first two hours behind.
#
# Off Colab this whole cell is inert, so the notebook stays runnable locally.
# ======================================================================================

import shutil

MIRROR_TO_DRIVE = True                       # False disables mirroring entirely
DRIVE_FOLDER    = "3DVG-results"             # under MyDrive/

DRIVE_ROOT = None

def mount_drive():
    """Mount Drive and return the destination directory, or None if unavailable."""
    global DRIVE_ROOT
    if not (MIRROR_TO_DRIVE and IN_COLAB):
        print("Drive mirroring off"
              + ("" if MIRROR_TO_DRIVE else " (MIRROR_TO_DRIVE=False)")
              + ("" if IN_COLAB else " -- not running on Colab, nothing to mirror"))
        return None
    try:
        from google.colab import drive
        mount = Path("/content/drive")
        if not (mount / "MyDrive").is_dir():
            drive.mount(str(mount))
        DRIVE_ROOT = mount / "MyDrive" / DRIVE_FOLDER
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted -> {DRIVE_ROOT}")
        print("outputs/ will be copied there after every experiment.")
    except Exception as error:
        # A failed mount must not take the run with it: the experiments are the point,
        # the backup is insurance.
        DRIVE_ROOT = None
        print(f"Drive not mounted ({error}). The run continues; results stay on this "
              f"machine only and will be lost when the runtime recycles.")
    return DRIVE_ROOT

def mirror_to_drive(quiet=False):
    """Copy outputs/ to Drive. Cheap to call repeatedly: only newer files are copied."""
    if DRIVE_ROOT is None:
        return
    source = REPO / "outputs"
    if not source.is_dir():
        return
    copied = 0
    for path in source.rglob("*"):
        if path.is_dir() or "tensorboard" in path.parts:
            continue
        target = DRIVE_ROOT / "outputs" / path.relative_to(source)
        try:
            if target.exists() and target.stat().st_mtime >= path.stat().st_mtime:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
            copied += 1
        except Exception as error:
            if not quiet:
                print(f"  could not copy {path.name}: {error}")
    if not quiet:
        print(f"mirrored {copied} new/changed file(s) to {DRIVE_ROOT}/outputs")
    return copied

mount_drive()


## 7 — run

One cell per stage so progress is visible and a failure is localised. Each experiment
prints what it is testing before it runs, streams its output live, and writes a report to
`outputs/reports/<experiment>/` — the full log, a `result.json`, and a readable
`report.md`. A non-zero exit is recorded and the run continues, because one broken
experiment should not abort the other nine.


In [ ]:
# ======================================================================================
# Runner: execute, narrate, report, mirror.
#
# Every experiment gets, in this order:
#   1. a header saying what it is testing and which reviewer comment it answers,
#   2. its command's live output,
#   3. a report written to outputs/reports/<key>/ -- the full log, a machine-readable
#      result.json, and a human-readable report.md,
#   4. a copy of everything under outputs/ pushed to Drive, if mounted.
#
# The report exists because a Colab runtime is disposable: the scrollback is gone the
# moment the tab closes, and the exit code alone does not say what a run was for.
#
# Two entry points, because phase A is used differently from phases B and C:
#   run_single(key)  -- one experiment, one cell. Used for every phase-A arm, so each
#                       retraining run can be started, skipped or resumed on its own.
#   run_stage(stage) -- every experiment in a stage. Used where the members are cheap
#                       and always wanted together (the analyses, the diagnostics).
# ======================================================================================

REPORTS = REPO / "outputs" / "reports"

STAGE_PURPOSE = {
    # ---- phase A: retraining ----------------------------------------------------------
    "train_no_copypaste": "Train the main configuration with proposal copy-paste "
                          "DISABLED. Everything else is held fixed, so the gap against "
                          "the reference arm is the augmentation's isolated "
                          "contribution (R2, R4.7).",
    "train_parser_gpt":   "Variant A, the reference arm: GPT-4o-mini parses, the "
                          "paper's main configuration. Every other parser arm is read "
                          "as a difference from this one.",
    "train_parser_spacy": "Variant B: a rule-based spaCy dependency parser instead of "
                          "an LLM. Answers R3.1 -- could a conventional parser do the "
                          "same job?",
    "train_parser_llama": "Variant C: open LLaMA-3 instead of the GPT API. Tests "
                          "whether the method depends on a proprietary model (R2, R3.1).",
    "train_parser_none":  "Variant D: no parser at all, architecture untouched. The "
                          "paper's Table 6 removed the parse AND the fusion modules "
                          "together; this removes only the parse, so the difference is "
                          "attributable to the parser alone (R4.4).",
    "train_parser_smalllm": "Variant E: a 0.5B local model. If it lands near variant A "
                            "the API leaves the critical path entirely (R3.1, R4.5).",
    "train_seeds":        "Repeat the main configuration under extra random seeds. "
                          "Several ablation gaps are below one percentage point; "
                          "without a spread they cannot be called stable (R4.8).",
    "train_attention_sweep": "Hyper-parameter sweep over the number of attention "
                             "layers (R3.3). Retrains, but answers a design question "
                             "rather than ablating a component.",
    # ---- phase B: evaluation ----------------------------------------------------------
    "eval_main":          "Run the trained checkpoint over val and write predictions.p, "
                          "which every phase-C analysis reads.",
    "eval_corruption":    "Feed deliberately corrupted parses to a trained model: how "
                          "gracefully does accuracy degrade as parse quality drops (R4.3)?",
    "eval_parser_swap":   "Swap the parser at evaluation time only, with training held "
                          "fixed -- separates train-time from test-time dependence (R4.4).",
    # ---- phase C: post-processing -----------------------------------------------------
    "parse_caches": "Build one parse cache per parser variant (A-E). Each is written to "
                    "disk once and reused offline -- no LM runs during training.",
    "scene_cache":  "Freeze the detector: run PointNet++/VoteNet/DETR once per scene and "
                    "store the proposals, so ablations train only the fusion head.",
    "parser_acc":   "How often does each parser recover the ground-truth target noun? "
                    "Isolates parser quality from grounding quality (R3.1).",
    "analyses":     "Turn predictions into the manuscript's tables and figures. CPU only.",
    "complexity":   "FLOPs, peak memory, latency -- the cost side of the trade-off (R2).",
    "diagnostics":  "Sanity checks on the cache, the distance bias, and the cached path.",
}

def describe(exp):
    """The header printed before a command runs."""
    device = "GPU" if exp["device"] == "gpu" else "CPU"
    produces = exp["produces"] or ["(no declared output file)"]
    phase = PHASE_OF.get(exp["stage"], "?")
    lines = [
        "=" * 88,
        f"EXPERIMENT   {exp['key']}",
        f"phase        {phase} -- {PHASE_NAME.get(phase, '')}",
        f"stage        {exp['stage']}   |   requires {device}",
        f"purpose      {exp['note'] or '--'}",
        f"script       {exp['script']}",
        f"produces     {produces[0]}",
    ]
    lines += [f"             {p}" for p in produces[1:]]
    return "\n".join(lines)

def write_report(exp, record, log_text):
    """One folder per experiment: raw log, structured result, and a readable summary."""
    folder = REPORTS / exp["key"]
    folder.mkdir(parents=True, exist_ok=True)

    (folder / "log.txt").write_text(log_text)

    phase = PHASE_OF.get(exp["stage"], "?")
    record = dict(record)
    record.update(key=exp["key"], stage=exp["stage"], phase=phase, device=exp["device"],
                  purpose=exp["note"], script=exp["script"],
                  produces=exp["produces"], finished_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                  environment="Google Colab" if IN_COLAB else "local",
                  gpu=GPU_NAME or "none")
    (folder / "result.json").write_text(json.dumps(record, indent=2))

    status = ("SUCCESS" if record.get("exit_code") == 0 else
              f"REFUSED -- {record['refused']}" if record.get("refused") else
              f"FAILED (exit {record.get('exit_code')})")
    outputs_present = [f"  - {p}  {'OK' if _glob.glob(str(REPO / p)) else 'MISSING'}"
                       for p in exp["produces"]] or ["  (none declared)"]
    tail = log_text.strip().splitlines()[-25:]

    (folder / "report.md").write_text("\n".join([
        f"# {exp['key']}", "",
        f"**{status}** in {record.get('elapsed_sec', 0)}s "
        f"on {record['finished_at']} ({record['environment']}, GPU: {record['gpu']})", "",
        f"Phase {phase} ({PHASE_NAME.get(phase, '')}), stage `{exp['stage']}`.", "",
        "## What this experiment tests", "",
        exp["note"] or "_no description recorded_", "",
        STAGE_PURPOSE.get(exp["stage"], ""), "",
        "## How it was run", "", "```", record.get("cmd", ""), "```", "",
        "## Declared outputs", "", *outputs_present, "",
        "## Last lines of output", "", "```", *tail, "```", "",
        "_Full output in `log.txt`; structured fields in `result.json`._",
    ]))
    return folder

def run_experiment(exp):
    action = PLAN.get(exp["key"], "SKIP-off")
    if action != "RUN":
        RESULTS[exp["key"]] = {"action": action}
        print(f"[{exp['key']}] {action}  ({exp['note']})")
        return

    args = list(exp["args"])
    if exp["device"] == "gpu" and not GPU_USABLE and FORCE_GPU:
        # Keep the machine alive: shrink any batch size the command carries.
        if "--batch_size" in args:
            args[args.index("--batch_size") + 1] = str(MIN_BATCH_SIZE)
        else:
            args += ["--batch_size", str(MIN_BATCH_SIZE)]

    cmd = [sys.executable, exp["script"], *args]
    printable = " ".join(cmd)
    print("\n" + describe(exp))
    print(f"$ {printable}\n{'-' * 88}")

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = GPU_ID if (DEVICE == "cuda" and GPU_USABLE) else ""
    env["PYTHONUNBUFFERED"] = "1"

    started = time.time()
    captured = []
    process = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        captured.append(line)
    code_ = process.wait()
    elapsed = time.time() - started

    refused = exp.get("refuses", {}).get(code_)
    record = {"action": "RUN", "exit_code": code_, "refused": refused,
              "elapsed_sec": round(elapsed, 1), "cmd": printable}
    RESULTS[exp["key"]] = record

    if code_ == 0:
        verdict = "OK"
    elif refused:
        verdict = f"REFUSED -- {refused}"
    elif code_ in (247, -9, 137):
        # 247/-9/137 are all SIGKILL: the host ran out of RAM, not the GPU.
        verdict = (f"FAILED (exit {code_}) -- killed by the OS, out of host RAM. "
                   f"Check LAZY_LANG_DATA is True.")
    else:
        verdict = f"FAILED (exit {code_})"

    folder = write_report(exp, record, "".join(captured))
    print(f"{'-' * 88}\n[{exp['key']}] {verdict} in {elapsed:.1f}s")
    print(f"           report -> {folder.relative_to(REPO)}/report.md")
    mirror_to_drive(quiet=True)

def _banner(title, subtitle=""):
    print(f"\n{'#' * 88}")
    print(f"# {title}")
    if subtitle:
        for line in textwrap.wrap(subtitle, 84):
            print(f"# {line}")
    print("#" * 88)

def run_single(key):
    """Run exactly one experiment. This is the phase-A entry point.

    Each retraining arm gets its own cell calling this, so the seven of them can be
    started, skipped and resumed independently -- which matters when one arm is hours
    of GPU time and a Colab session can disappear halfway through.
    """
    matches = [e for e in EXPERIMENTS if e["key"] == key]
    if not matches:
        print(f"[{key}] no such experiment -- check the key against the registry cell.")
        return
    exp = matches[0]
    _banner(f"{key}   (phase {PHASE_OF.get(exp['stage'], '?')}, "
            f"stage {exp['stage']})", STAGE_PURPOSE.get(exp["stage"], ""))
    run_experiment(exp)

def run_stage(stage):
    """Run every experiment in a stage. Used for phases B and C."""
    chosen = [e for e in EXPERIMENTS if e["stage"] == stage]
    planned = [e for e in chosen if PLAN.get(e["key"]) == "RUN"]
    _banner(f"STAGE: {stage}   (phase {PHASE_OF.get(stage, '?')}; "
            f"{len(planned)} of {len(chosen)} experiment(s) will run)",
            STAGE_PURPOSE.get(stage, ""))
    for exp in chosen:
        run_experiment(exp)
    tally = {}
    for exp in chosen:
        action = RESULTS.get(exp["key"], {}).get("action", PLAN.get(exp["key"], "SKIP"))
        tally[action] = tally.get(action, 0) + 1
    print(f"\n# stage '{stage}' finished: "
          + "  ".join(f"{k}={v}" for k, v in sorted(tally.items())))

def run_phase(phase):
    """Run every stage in a phase, in registration order."""
    seen = []
    for exp in EXPERIMENTS:
        if PHASE_OF.get(exp["stage"]) == phase and exp["stage"] not in seen:
            seen.append(exp["stage"])
    print(f"\n{'=' * 88}")
    print(f"PHASE {phase} -- {PHASE_NAME.get(phase, '')}   ({len(seen)} stage(s))")
    print("=" * 88)
    for stage_key in seen:
        run_stage(stage_key)

print("runner ready -- reports go to outputs/reports/<experiment>/")
print("  run_single(key)  one experiment      (phase A uses this)")
print("  run_stage(stage) one stage           (phases B and C use this)")
print("  run_phase(p)     every stage in a phase, for 'A', 'B' or 'C'")

### Applying the training configuration

The phase-A runners are **self-contained by design** — each carries its own `CONFIG`
block so that changing one experiment can never disturb another. That is the right
property for the runners and the wrong one for a notebook, where you want a single place
to set the batch size.

This cell reconciles the two: it rewrites the `CONFIG` values in the seven runner files
from the notebook's settings above. It only touches assignments it recognises, prints
every change it makes, and leaves everything else in the files untouched.

Run it after editing the configuration cell and before running any phase-A experiment.

In [ ]:
# ======================================================================================
# Push the notebook's training configuration into the phase-A runner files.
#
# Rewrites only whole-line assignments of known names, at column 0, in the CONFIG block
# of each runner -- a regex anchored to `^NAME = ...` cannot touch a docstring, a comment
# or a nested value. Anything it changes is printed, so a surprising edit is visible
# rather than silent.
# ======================================================================================

import ast
import re as _re

RUNNER_DIR = REPO / "experiments" / "ablation" / "runners"

PHASE_A_RUNNERS = [
    "run_no_copypaste.py", "run_parser_gpt.py", "run_parser_spacy.py",
    "run_parser_llama.py", "run_parser_none.py", "run_parser_smalllm.py",
    "run_seeds.py",
]

# name in the runner -> value from this notebook
TRAINING_CONFIG = {
    "EPOCH":           EPOCH,
    "BATCH_SIZE":      BATCH_SIZE,
    "VAL_STEP":        VAL_STEP,
    "VERBOSE":         VERBOSE,
    "LR":              LR,
    "COSLR":           COSLR,
    "LANG_NUM_MAX":    LANG_NUM_MAX,
    "WARM_START":      WARM_START,
    "WARM_START_FROM": WARM_START_FROM,
    "FUSION_VARIANT":  WARM_START_FUSION_VARIANT,
    "NUM_WORKERS":     NUM_WORKERS,
    "PREFETCH_FACTOR": PREFETCH_FACTOR,
    "LAZY_LANG_DATA":  LAZY_LANG_DATA,
    "EXTRA_ARGS":      EXTRA_ARGS,
    "CACHED_SCENES_ROOT": CACHED_SCENES_ROOT,
    "USE_CACHED_SCENES":  USE_CACHED_SCENES,
}

# SEEDS belongs only to the seed runner; the others carry a scalar SEED.
PER_RUNNER = {"run_seeds.py": {"SEEDS": SEEDS}}

def apply_training_config(dry_run=False):
    """Rewrite the CONFIG assignments in each phase-A runner. Returns a change count."""
    changed_total = 0
    for filename in PHASE_A_RUNNERS:
        path = RUNNER_DIR / filename
        if not path.is_file():
            print(f"  {filename:<24} MISSING -- skipped")
            continue

        text = original = path.read_text()
        wanted = dict(TRAINING_CONFIG)
        wanted.update(PER_RUNNER.get(filename, {}))
        changes = []

        for name, value in wanted.items():
            pattern = _re.compile(rf"^({name})( *= *)(.+?)( *(?:#.*)?)$", _re.M)
            match = pattern.search(text)
            if not match:
                continue                      # this runner does not carry that knob
            current = match.group(3).strip()
            replacement = repr(value)
            # Compare by value, not by text: repr() normalises "x" to 'x', and a
            # quote-style difference is not a change worth rewriting the file for.
            try:
                if current and ast.literal_eval(current) == value:
                    continue
            except (ValueError, SyntaxError):
                pass                          # not a literal -- fall through and replace
            if current == replacement:
                continue
            text = pattern.sub(
                lambda m: f"{m.group(1)}{m.group(2)}{replacement}{m.group(4)}", text, count=1)
            changes.append(f"{name}: {current} -> {replacement}")

        if changes and not dry_run:
            path.write_text(text)
        status = "would change" if dry_run else "updated"
        if changes:
            print(f"  {filename:<24} {status}: " + "; ".join(changes))
            changed_total += len(changes)
        else:
            print(f"  {filename:<24} already matches")
    return changed_total

print("applying the notebook's training configuration to the phase-A runners")
print(f"  epochs={EPOCH}  batch={BATCH_SIZE}  val_step={VAL_STEP}  lr={LR}  "
      f"warm_start={WARM_START}  workers={NUM_WORKERS}\n")
_n = apply_training_config()
print(f"\n{_n} value(s) changed.")
if WARM_START:
    _ckpt = REPO / "outputs" / str(WARM_START_FROM) / "model_criteria_25.pth"
    print(f"\nwarm start: outputs/{WARM_START_FROM}/model_criteria_25.pth "
          f"{'FOUND' if _ckpt.is_file() else 'MISSING -- training would fail'}")
    print(f"            fusion variant {WARM_START_FUSION_VARIANT!r}")
    if not _ckpt.is_file():
        print("            fix WARM_START_FROM, or set WARM_START=False to train "
              "from scratch.")
else:
    print("\nwarm start OFF -- every arm trains from random initialisation "
          "(slower; expect to need more epochs).")

---

# Phase A — retraining

**GPU, hours per experiment.** These seven train a model from scratch and are what the
revision's ablation table is built from. Everything in phases B and C is cheap by
comparison; this is the part that takes a week of GPU time.

Each arm below has **its own cell and its own switch** (`RUN_TRAIN_*` in the config
cell). They are independent on purpose: if a Colab session dies during A4, turn A1–A3
off and re-run — the finished arms are not repeated, and nothing is lost.

All seven train **only the fusion head**: under the frozen-detector protocol the
detector runs once, offline, and its proposals are read from `cached_scenes/`. That is
what makes a controlled comparison possible — the arms differ in exactly one thing each,
so a gap between two of them is attributable to that thing and not to detector noise.

> **Order matters only for A2.** The parser arms are compared *against* A2 (variant A,
> the paper's configuration), so if you are running a subset, keep A2 in it.

### A1 — copy-paste augmentation off

**Ablates:** the proposal copy-paste augmentation. **Reviewer:** R2, R4.7.

Reviewer #2 called copy-paste "an interesting idea" whose "independent contribution
should be analyzed through a dedicated ablation"; R4.7 makes the same point. The paper
never separated it from LGA and the fusion modules.

The augmentation lives *inside the fusion network* — `models/match_module.py`, the two
blocks gated on `istrain and random_numer < 0.5`, which copy valid proposals' features
into invalid slots. Because that is downstream of the scene cache it keeps randomising
normally under `--use_cached_scenes`, so this ablation needs no end-to-end bypass.

Everything else is identical to A2, so the gap between them is the augmentation alone.

In [ ]:
# switch: RUN_TRAIN_NO_COPYPASTE   (config cell, section "PHASE A")
run_single("train-no-copypaste")

### A2 — variant A: GPT-4o-mini (reference arm)

**Ablates:** nothing — this *is* the paper's main configuration. **Reviewer:** R4.4.

The reference arm. A3–A6 each swap out the parser and are read as a difference from
this run, so it is the one arm you cannot skip if you intend to compare anything.

Reads `data_parsing/final_parsing_tokenized`, which already ships with the repo — no
API call is made here or anywhere else in training. Parsing is entirely offline.

In [ ]:
# switch: RUN_TRAIN_PARSER_GPT   (config cell, section "PHASE A")
run_single("train-parser-gpt")

### A3 — variant B: rule-based spaCy

**Ablates:** the LLM, replaced by a rule-based dependency parser. **Reviewer:** R3.1.

Reviewer #3 asked directly whether "similar performance could be achieved using
conventional dependency parsers." This is that experiment: a spaCy dependency parse
feeds the same fusion network, with everything else held fixed.

If A3 lands close to A2, the LLM is not carrying the method — which would be a finding
worth reporting honestly rather than burying.

**Needs** `data_parsing/spacy_parsing_tokenized/` — built in phase C1 (~1 min, CPU).

In [ ]:
# switch: RUN_TRAIN_PARSER_SPACY   (config cell, section "PHASE A")
run_single("train-parser-spacy")

### A4 — variant C: LLaMA-3

**Ablates:** the proprietary API, replaced by an open model. **Reviewer:** R2, R3.1.

Tests whether the method depends on GPT specifically or just on *some* competent parser.
An open model matching GPT-4o-mini removes the API from the reproducibility story.

**Needs** `data_parsing/llama_parsing_tokenized_clipped/`. Note the *clipped*: the raw
LLaMA cache violates the 7/17/75 token caps in 5 annotations, each of which raises
inside `pack_padded_sequence` mid-training. `clip_parse_cache.py` writes a corrected
copy and leaves the original untouched. The runner checks for this and refuses with an
explanation rather than crashing three hours in.

In [ ]:
# switch: RUN_TRAIN_PARSER_LLAMA   (config cell, section "PHASE A")
run_single("train-parser-llama")

### A5 — variant D: no parser

**Ablates:** the parser entirely, architecture untouched. **Reviewer:** R4.4.

The paper's Table 6 removed the parsed language input *and* the fusion sub-modules
together, so it could not separate the LLM's contribution from A2F/TAF's. This run
separates them: A2F and TAF still run, only the parse content is gone. The gap against
A2 is the parser's contribution and nothing else.

**Needs** `data_parsing/noparse_tokenized/`, where all three fields carry the single
token `unk` (~5 s, CPU, phase C1). A literal zero mask is not reachable without editing
`lib/dataset.py` — `pack_padded_sequence` rejects zero-length sequences and no GloVe
token maps to a zero row. **State this encoding in the manuscript**; it is the honest
description of what "no parser" means here.

In [ ]:
# switch: RUN_TRAIN_PARSER_NONE   (config cell, section "PHASE A")
run_single("train-parser-none")

### A6 — variant E: small local LM

**Ablates:** model scale — GPT-4o-mini replaced by a 0.5B local model.
**Reviewer:** R3.1, R4.5.

Variant B covers the rule-based end and A2 the frontier end; this covers the middle. A
0.5B model landing near GPT-4o-mini is the most useful outcome available here: it
removes the API from the critical path and answers the cost objection in R2/R4.5 at the
same time.

**Needs** `data_parsing/smalllm_parsing_tokenized/` — phase C1, **GPU, ~80 min on a
T4**. Generated once, written to disk, then reused offline; that is why `OVERRIDE_EXP`
defaults to False. The generator also reports the **malformed-output rate**, which the
manuscript should quote — R4.3 asks what happens when the model returns something
unusable.

In [ ]:
# switch: RUN_TRAIN_PARSER_SMALLLM   (config cell, section "PHASE A")
run_single("train-parser-smalllm")

### A7 — seed repeats

**Ablates:** nothing — repeats A2 under additional random seeds. **Reviewer:** R4.8.

Reviewer #4: *"The ScanRefer ablations are reported without variation across runs,
although several differences are below one percentage point. Results over multiple
random seeds would help establish whether these differences are stable."*

Runs the unchanged main configuration once per seed, each into its own output folder, so
the spread can be reported as mean ± std beside the ablation table. Without this, every
sub-point gap in the paper is unfalsifiable.

What the seed still controls under the frozen detector: geometric point-cloud
augmentation is off, but proposal copy-paste, the word masking and sentence reversal in
`LangModule`, weight initialisation and batch order all sit downstream of the cache and
keep randomising. Those are the variance sources R4.8 is asking about.

Aggregate the runs afterwards with `seed-aggregate` in phase C4.

> **Warm start and this experiment.** A7 is the one arm where the shared warm start
> deserves a second thought. All seven arms fine-tune from the same checkpoint, which
> means the seed spread measured here is the spread *of fine-tuning*, not of training
> from scratch — it will be **narrower** than the from-scratch spread R4.8 is really
> asking about, because the runs start from a common point.
>
> That is still a valid and reportable number, but it must be described accurately:
> "variation across seeds when fine-tuning from a shared initialisation." If you want
> the wider, more conservative figure, set `WARM_START = False` in the configuration
> cell, re-run the apply cell, and expect to raise `EPOCH` — from random init, 50 epochs
> will not be enough.

In [ ]:
# switch: RUN_TRAIN_SEEDS   (config cell, section "PHASE A")
run_single("train-seeds")

### A8 — attention-layer sweep

**Not an ablation arm** — a hyper-parameter sweep that happens to retrain.
**Reviewer:** R3.3.

Kept apart from A1–A7 because it answers a design question (how many attention layers
should the fusion head have?) rather than isolating a component's contribution. Verified
on CPU that depth 1→4 scales the module as expected before committing GPU time.

In [ ]:
# switch: RUN_TRAIN_ATTENTION_SWEEP   (config cell, section "PHASE A")
run_single("train-attention-sweep")

---

# Phase B — evaluation

**GPU, minutes.** Builds a model and runs it, but **trains nothing**. Everything here
needs a checkpoint that already exists — either one of phase A's, or the pre-trained
`CHECKPOINT` set in the config cell.

B1 comes first because it writes `predictions.p`, which most of phase C reads.

### B1 — main evaluation → `predictions.p`

Runs `CHECKPOINT` over the validation split and writes `predictions.p`. Almost every
analysis in phase C reads this file, so run it before C4.

Built with `CHECKPOINT_FUSION_VARIANT` (`"original"` by default): the shipped checkpoint
was trained with the older fusion head, and loading it into today's `MatchModule` would
leave 178 tensors randomly initialised. The hooks refuse that outright rather than
reporting a meaningless accuracy — a refusal here is the system working, not a bug.

In [ ]:
# switch: RUN_EVAL_MAIN
run_stage("eval_main")

### B2 — parse-corruption sweep

**Reviewer:** R4.3. Feeds deliberately corrupted parses to an already-trained model at
10 %, 25 % and 50 % corruption, one val pass each, and archives the predictions.

This measures **error propagation**: how gracefully does grounding degrade as parse
quality drops? A method that collapses at 10 % corruption is fragile in a way the main
table would never reveal. C4's `error-propagation` turns these archives into the curve.

Training is untouched — only the test-time input changes.

In [ ]:
# switch: RUN_EVAL_CORRUPTION
run_stage("eval_corruption")

### B3 — evaluation-only parser swap

**Reviewer:** R4.4. Takes the GPT-trained model and feeds it a *different* parser's
output at test time, with training held completely fixed.

Separates two things the parser arms in phase A necessarily conflate: dependence on the
parser **during training** versus **at inference**. A model that tolerates a swapped
parser at test time is relying on the parse far less than one that does not.

In [ ]:
# switch: RUN_EVAL_PARSER_SWAP
run_stage("eval_parser_swap")

---

# Phase C — post-processing

**Mostly CPU, seconds to minutes.** Reads files off disk and writes reports. No model is
built and nothing is trained, so this phase runs on any machine — including one with no
GPU at all.

Two exceptions carry a GPU tag: the variant-E parse cache in C1 and the scene cache in
C2. Both are one-off generation steps, not analysis, and both are skipped automatically
without a usable GPU.

**C1 produces the parse caches that phase A consumes**, so on a fresh machine the real
order is C1 → A → B → the rest of C. It is listed last because that is where it belongs
conceptually, and because the caches usually already exist.

### C1 — parse caches (the input to phase A)

One cache per parser variant, each written to disk once and then reused offline. **No
language model runs during training or evaluation** — this is the step that makes that
true, and it is why parsing contributes zero to per-query inference latency.

| | variant | cost |
|---|---|---|
| `parse-spacy` | B — rule-based | ~1 min, CPU |
| `parse-none` | D — all fields `unk` | ~5 s, CPU |
| `parse-corrupt` | corrupted val sets for B2 | seconds, CPU |
| `parse-smalllm` | E — 0.5B local model | **~80 min, GPU** |

Variant A (GPT-4o-mini) and variant C (LLaMA) ship with the repo and are not regenerated
here; C only needs the one-off clipping step described in A4.

In [ ]:
# switch: RUN_POST_PARSE_CACHES
run_stage("parse_caches")

### C2 — scene cache (frozen detector) — *off by default*

Runs PointNet++ → Hough voting → DETR once per scene and stores the proposals, so every
phase-A arm trains only the fusion head.

**Off by default, deliberately.** `cached_scenes.zip` already contains a complete cache
(`meta.json: complete=true`, 562 train + all val scenes) built from the same detector
checkpoint. Turning this on spends GPU hours reproducing bytes that are already on disk.
Enable it only to regenerate the cache from scratch.

The second experiment here is the strict correctness test: an end-to-end forward pass
versus the cached path, which must agree to 1e-4.

In [ ]:
# switch: RUN_POST_SCENE_CACHE
run_stage("scene_cache")

### C3 — parser target accuracy

**Reviewer:** R3.1. How often does each parser recover the ground-truth target noun?

This isolates **parser quality** from **grounding quality**. Without it, a weak result
for variant B is ambiguous — bad parses, or good parses the fusion network cannot use?
C3 answers the first half directly, one number per variant, no model involved.

Worth reading next to the manual annotation sheets from C4: automatic target accuracy
misses synonym mismatches (`couch → sofa`, `refrigerator → fridge`), which the
hand-annotated sample catches.

In [ ]:
# switch: RUN_POST_PARSER_ACC
run_stage("parser_acc")

### C4 — analyses → the paper's tables and figures

The reporting layer. Reads `predictions.p` (from B1) and the archives from B2/A7, and
writes the manuscript's tables, figures and markdown reports.

| experiment | produces | reviewer |
|---|---|---|
| `results-table` | the main results table, LaTeX included | R1.4, R3.2, R4.9 |
| `linguistic-complexity` | accuracy split by description complexity | R4.2 |
| `parse-quality-split` | grounding accuracy where the parse was right vs wrong | R4.3 |
| `failure-cases` + `failure-figures` | the qualitative failure figure | R2, R4 |
| `annotation-sheet` | 200-sample manual annotation CSVs | R1.2, R4.3 |
| `error-propagation` | the degradation curve — **needs B2** | R2, R4.3 |
| `seed-aggregate` | mean ± std across seeds — **needs A7** | R4.8 |

The last two are the ones that silently produce nothing if their upstream phase was
skipped, so the preflight table flags them as `SKIP-input` rather than letting them
write an empty report.

In [ ]:
# switch: RUN_POST_ANALYSES
run_stage("analyses")

### C5 — complexity: FLOPs, memory, latency

**Reviewer:** R2, R4.5 — the cost side of the trade-off, which the paper reported only
partially.

`parsing-offline-check` is the one to read first: it *verifies* that no parser is
invoked in the forward path, which is the claim the latency numbers depend on. The other
two measure per-query parse cost and the model's own FLOPs, peak memory and latency
(mean/std/p50/p95, not a bare mean).

In [ ]:
# switch: RUN_POST_COMPLEXITY
run_stage("complexity")

### C6 — diagnostics: does the implementation match the paper?

Correctness checks rather than results — the experiments that answer "is the thing we
described the thing we ran?"

- **`verify-eq7`** (R1.3, R4.6) — the sign of the distance bias in Eq. 7. This one
  found a real typesetting defect in the manuscript.
- **`audit-scene-cache`** — model-free cache integrity plus the recall ceiling the
  frozen detector imposes. Every phase-A number is bounded by that ceiling, so it
  belongs in the paper.
- **`cached-eval-cpu`** — accuracy through the cached path without CUDA. Refuses with
  exit 2 if the checkpoint does not match the fusion variant, which is reported as
  `REFUSED`, not as a failure.

In [ ]:
# switch: RUN_POST_DIAGNOSTICS
run_stage("diagnostics")

### After the annotation sheets are filled in

`annotation_sheet.py` writes CSVs with blank columns. Once a human has filled them in (see
`outputs/analysis/annotation/instructions.md`), list them here and run the tally.

In [ ]:
FILLED_SHEETS = {}   # e.g. {"gpt4o-mini": "outputs/analysis/annotation/annotation_sheet_gpt4o-mini.csv"}

if FILLED_SHEETS:
    args = []
    for label, path in FILLED_SHEETS.items():
        args += ["--sheet", f"{label}={path}"]
    run_experiment({"key": "error-taxonomy", "stage": "analyses", "device": "cpu",
                    "script": "experiments/analysis/error_taxonomy.py",
                    "args": args, "produces": [], "note": ""})
    PLAN["error-taxonomy"] = "RUN"
else:
    print("FILLED_SHEETS is empty -- annotate the CSVs under "
          "outputs/analysis/annotation/ first, then list them above.")

## Summary

What ran, what it produced, and what is still outstanding — grouped by phase, so an
incomplete run tells you which phase to resume rather than which stage number to
remember.

In [ ]:
print("=" * 96)
print("EXECUTION SUMMARY")
print("=" * 96)

ok = failed = skipped = refused = 0
for phase in ("A", "B", "C"):
    chosen = [e for e in EXPERIMENTS if PHASE_OF.get(e["stage"]) == phase
              and e["key"] in RESULTS]
    if not chosen:
        continue
    print(f"\n  phase {phase} -- {PHASE_NAME.get(phase, '')}")
    for exp in chosen:
        entry = RESULTS[exp["key"]]
        if entry.get("action") != "RUN":
            print(f"    {entry.get('action', 'SKIP'):<11} {exp['key']}")
            skipped += 1
        elif entry.get("exit_code") == 0:
            print(f"    {'OK':<11} {exp['key']:<26} {entry['elapsed_sec']:>8.1f}s")
            ok += 1
        elif entry.get("refused"):
            print(f"    {'REFUSED':<11} {exp['key']:<26} {entry['refused']}")
            refused += 1
        else:
            print(f"    {'FAILED':<11} {exp['key']:<26} exit {entry['exit_code']}")
            failed += 1

print(f"\n  {ok} ok, {failed} failed, {refused} refused, {skipped} skipped")
if refused:
    print("  REFUSED means the script checked a precondition and declined on purpose; "
          "scroll up for its reason.")
if not RESULTS:
    print("  nothing has been run yet -- execute the phase cells above.")

# ---- how much of phase A is actually done? -------------------------------------------
# The ablation table cannot be written until these seven exist, so they get their own
# tally rather than being averaged into the total above.
print("\n" + "=" * 96)
print("PHASE A -- RETRAINING PROGRESS  (the ablation table depends on these)")
print("=" * 96)
RETRAIN_KEYS = [e["key"] for e in EXPERIMENTS
                if PHASE_OF.get(e["stage"]) == "A" and e["key"] != "train-attention-sweep"]
done_a = 0
for key in RETRAIN_KEYS:
    entry = RESULTS.get(key, {})
    if entry.get("action") == "RUN" and entry.get("exit_code") == 0:
        mark, done_a = "trained", done_a + 1
    elif entry.get("action") == "RUN":
        mark = f"FAILED (exit {entry.get('exit_code')})"
    elif entry.get("action"):
        mark = entry["action"]
    else:
        mark = "not reached in this session"
    print(f"  {key:<26} {mark}")
print(f"\n  {done_a} of {len(RETRAIN_KEYS)} retraining arms completed in this session.")

print("\n" + "=" * 96)
print("ARTIFACTS AND THE PAPER SECTION THEY FEED")
print("=" * 96)
for pattern, purpose in [
    ("outputs/analysis/results_table/results_table.md", "main results table (R1.4, R3.2, R4.9)"),
    ("outputs/analysis/linguistic_complexity/linguistic_complexity.*", "accuracy by linguistic complexity (R4.2)"),
    ("outputs/analysis/parse_quality_split/parse_quality_split.md", "parse correctness vs grounding (R4.3)"),
    ("outputs/analysis/parse_error_propagation/parse_error_propagation.md", "error propagation (R2, R4.3)"),
    ("outputs/analysis/failure_cases/failure_cases.md", "qualitative failures (R2, R4)"),
    ("experiments/analysis/figures/*.png", "the qualitative figure (R2, R4)"),
    ("outputs/analysis/annotation/annotation_sheet_*.csv", "manual parse annotation (R1.2, R4.3)"),
    ("outputs/parser_eval/target_accuracy_*.json", "parser quality layer 1 (R1.2, R4.3)"),
    ("outputs/complexity/complexity_report.json", "FLOPs / memory / latency (R2, R4.5)"),
    ("outputs/complexity/parsing_latency_report.json", "parsing cost reported separately (R4.5)"),
    ("outputs/diagnostics/eq7_distance_bias.json", "sign of Eq. 7 (R1.3, R4.6)"),
    ("outputs/diagnostics/scene_cache_audit_val.json", "scene-cache integrity"),
    ("outputs/ablation/attention_layer_sweep/sweep_summary.json", "attention-layer sweep (R3.3)"),
]:
    matches = _glob.glob(str(REPO / pattern))
    mark = "OK     " if matches else "missing"
    extra = f" ({len(matches)} files)" if len(matches) > 1 else ""
    print(f"  {mark} {pattern}{extra}\n          -> {purpose}")

print("\n" + "=" * 96)
print("STILL TO DO")
print("=" * 96)
todo = []
if not _glob.glob(str(REPO / "outputs" / "*ABL-PARSER*")):
    todo.append("phase A (GPU): the parser arms -- the ablation table needs all five")
if not _glob.glob(str(REPO / "outputs" / "*ABL-NO-COPYPASTE*")):
    todo.append("phase A (GPU): A1 no-copypaste -> the augmentation's isolated contribution")
if not _glob.glob(str(REPO / "outputs" / "*ABL-SEED*")):
    todo.append("phase A (GPU): A7 seed runs -> mean +/- std; R4.8 is unanswerable without them")
if not have(f"outputs/{CHECKPOINT}/corruption"):
    todo.append("phase B (GPU): B2 corruption sweep -> the error-propagation curve")
if not have("data_parsing", PARSERS["smalllm"], "tokenized_parsed_result_val.json"):
    todo.append("phase C1 (GPU): the small-LM parse cache -> variant E cannot train without it")
if not _glob.glob(str(REPO / "outputs/analysis/annotation/error_taxonomy.md")):
    todo.append("annotate the 200-sample sheets by hand, then run the tally cell")
if not GPU_USABLE:
    todo.append("every GPU phase: run this notebook on Colab, or install a torch build "
                "matching this card")
for item in todo or ["nothing outstanding"]:
    print(f"  - {item}")

summary_path = REPO / "outputs" / "run_analysis_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(
    {"in_colab": IN_COLAB, "device": DEVICE, "gpu_usable": GPU_USABLE, "gpu": GPU_NOTE,
     "force_gpu": FORCE_GPU, "override_exp": OVERRIDE_EXP,
     "checkpoint": CHECKPOINT, "cached_scenes": bool(CACHE_ARGS),
     "phases": PHASE_OF, "stages": RUN, "plan": PLAN, "results": RESULTS,
     "retraining_done": done_a, "retraining_total": len(RETRAIN_KEYS),
     "todo": todo}, indent=2))
print(f"\nwrote {summary_path.relative_to(REPO)}")

# ---- master report: one document describing the whole run -----------------------------
lines = [
    "# 3D Visual Grounding - experiment run report", "",
    f"Run finished {time.strftime('%Y-%m-%d %H:%M:%S')} on "
    f"{'Google Colab' if IN_COLAB else 'a local machine'} "
    f"(device: {DEVICE}, GPU: {GPU_NAME or 'none'}).", "",
    f"**{ok} succeeded, {failed} failed, {refused} refused, {skipped} skipped.** "
    f"Retraining arms completed this session: {done_a} of {len(RETRAIN_KEYS)}.", "",
    "## Configuration", "",
    f"- checkpoint: `{CHECKPOINT}` (fusion variant `{CHECKPOINT_FUSION_VARIANT}`)",
    f"- frozen-detector protocol: {'on' if CACHE_ARGS else 'off'}",
    f"- lazy language data: {LAZY_LANG_DATA} "
    f"(False needs ~36 GB of host RAM and is killed on Colab)",
    f"- re-run completed experiments: {OVERRIDE_EXP}", "",
]

for phase in ("A", "B", "C"):
    chosen = [e for e in EXPERIMENTS if PHASE_OF.get(e["stage"]) == phase]
    if not chosen:
        continue
    lines += [f"## Phase {phase} - {PHASE_NAME.get(phase, '')}", "",
              "| experiment | stage | status | time | what it tests |",
              "|---|---|---|---|---|"]
    for exp in chosen:
        entry = RESULTS.get(exp["key"], {})
        if entry.get("action") != "RUN":
            status = entry.get("action", "not reached")
        elif entry.get("exit_code") == 0:
            status = "OK"
        elif entry.get("refused"):
            status = "refused"
        else:
            status = f"failed ({entry['exit_code']})"
        seconds = entry.get("elapsed_sec")
        lines.append(f"| `{exp['key']}` | {exp['stage']} | {status} | "
                     f"{str(seconds) + 's' if seconds else '-'} | {exp['note']} |")
    lines.append("")

lines += ["## Outstanding", ""] + [f"- {item}" for item in (todo or ["nothing"])]
lines += ["", "Per-experiment logs and reports are under `outputs/reports/<experiment>/`."]

report_path = REPO / "outputs" / "reports" / "RUN_REPORT.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(lines))
print(f"wrote {report_path.relative_to(REPO)}")

# ---- final push: everything, including whatever the last experiment wrote --------------
if DRIVE_ROOT is not None:
    print("\nfinal mirror to Drive ...")
    n = mirror_to_drive()
    print(f"results are in {DRIVE_ROOT}/outputs and will outlive this runtime.")
else:
    print("\nNOT mirrored to Drive: everything above lives on this runtime only and is "
          "lost when it recycles. Mount Drive in the Drive section to keep it.")